# 05 - Modelling: MLP

Trains a small Multi-Layer Perceptron on the same 20-feature schema, as a deep-learning comparison point against Random Forest and XGBoost from `04_modelling_classical.ipynb`. Reproduces `04`'s exact split inline (same fixed `RANDOM_STATE`, same source `dataset_pe_v2_full.csv`, no split files saved to disk). Test set is not touched, model selection happens in `06_evaluation.ipynb`.

**Not executed in this sandbox** (no TensorFlow installed here). Run locally: `Kernel -> Restart & Run All`.


In [1]:
# Standard library
import sys

# Third-party
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Model, layers

pd.set_option("display.max_columns", None)
sys.path.append("../src")
from constants import ORDER_OF_FEATURES, LABEL_COLUMN, RANDOM_STATE

tf.random.set_seed(RANDOM_STATE)

**Compute-constrained downsample of the training pool.** `GridSearchCV`'s exhaustive search over the full ~600,000-row training pool is too slow on typical laptop hardware (each grid search trains many separate models across cross-validation folds). Downsamples `train_pool` to 75,000 rows per class (150,000 total) before the 85/15 train/validation split, using a fixed `RANDOM_STATE` so the reduction is reproducible. **The held-out `test` set (EMBER's own official 200,000-row split) is NOT reduced**: `06_evaluation.ipynb`'s final reported numbers still come from the complete, standard benchmark split, only the tuning step works on a smaller pool.


In [2]:
df = pd.read_csv("../data/dataset_pe_v2_full.csv")
train_pool = df[df["OriginalSplit"] == "train"].reset_index(drop=True)

TRAIN_POOL_SAMPLE_PER_CLASS = 75000
train_pool = pd.concat([
    g.sample(n=min(TRAIN_POOL_SAMPLE_PER_CLASS, len(g)), random_state=RANDOM_STATE)
    for _, g in train_pool.groupby(LABEL_COLUMN)
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print("downsampled train_pool:", train_pool.shape)

train_df = pd.concat([
    g.sample(frac=0.85, random_state=RANDOM_STATE)
    for _, g in train_pool.groupby(LABEL_COLUMN)
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
val_df = train_pool.drop(
    pd.concat([g.sample(frac=0.85, random_state=RANDOM_STATE) for _, g in train_pool.groupby(LABEL_COLUMN)]).index
).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X_train, y_train = train_df[ORDER_OF_FEATURES], train_df[LABEL_COLUMN]
X_val, y_val = val_df[ORDER_OF_FEATURES], val_df[LABEL_COLUMN]
print("train:", X_train.shape, "val:", X_val.shape)

downsampled train_pool: (150000, 23)
train: (127500, 20) val: (22500, 20)


## Scale the features

Fit once on `X_train` only, the same leakage-safe rule `04`'s pipelines enforce automatically; here it is manual since a neural network is not wrapped in a scikit-learn `Pipeline`.


In [3]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)

## Build the MLP

Input (20 features) -> 128 -> 64 -> 1 output (probability of malicious), each hidden layer followed by dropout for regularisation.


In [4]:
inputs = layers.Input(shape=(X_train_scaled.shape[1],))
x = layers.Dense(128, activation="relu")(inputs)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 20)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │           2,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 11,009 (43.00 KB)

 Trainable params: 11,009 (43.00 KB)

 Non-trainable params: 0 (0.00 B)

## Train the model

Classes are exactly balanced (50/50), so no `class_weight` correction is required. `EarlyStopping` halts training once validation performance stalls for 3 epochs and restores the best weights seen.


In [5]:
early_stop = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train.values,
    validation_data=(X_val_scaled, y_val.values),
    epochs=30, batch_size=256,
    callbacks=[early_stop],
    verbose=2,
)

Epoch 1/30
499/499 - 3s - 6ms/step - accuracy: 0.7627 - loss: 0.5125 - val_accuracy: 0.8050 - val_loss: 0.4281
Epoch 2/30
499/499 - 1s - 3ms/step - accuracy: 0.8053 - loss: 0.4388 - val_accuracy: 0.8278 - val_loss: 0.3934
Epoch 3/30
499/499 - 3s - 5ms/step - accuracy: 0.8189 - loss: 0.4051 - val_accuracy: 0.8362 - val_loss: 0.3634
Epoch 4/30
499/499 - 2s - 5ms/step - accuracy: 0.8277 - loss: 0.3836 - val_accuracy: 0.8420 - val_loss: 0.3492
Epoch 5/30
499/499 - 1s - 3ms/step - accuracy: 0.8328 - loss: 0.3699 - val_accuracy: 0.8480 - val_loss: 0.3371
Epoch 6/30
499/499 - 1s - 3ms/step - accuracy: 0.8382 - loss: 0.3610 - val_accuracy: 0.8507 - val_loss: 0.3297
Epoch 7/30
499/499 - 2s - 3ms/step - accuracy: 0.8413 - loss: 0.3518 - val_accuracy: 0.8553 - val_loss: 0.3236
Epoch 8/30
499/499 - 1s - 3ms/step - accuracy: 0.8441 - loss: 0.3466 - val_accuracy: 0.8593 - val_loss: 0.3179
Epoch 9/30
499/499 - 2s - 5ms/step - accuracy: 0.8472 - loss: 0.3398 - val_accuracy: 0.8633 - val_loss: 0.3112
E

## Evaluate on the validation set

In [6]:
val_preds = (model.predict(X_val_scaled) >= 0.5).astype(int).ravel()
print(classification_report(y_val, val_preds, target_names=["benign", "malicious"]))

704/704 ━━━━━━━━━━━━━━━━━━━━ 1s 776us/step
              precision    recall  f1-score   support

      benign       0.88      0.88      0.88     11250
   malicious       0.88      0.88      0.88     11250

    accuracy                           0.88     22500
   macro avg       0.88      0.88      0.88     22500
weighted avg       0.88      0.88      0.88     22500



## Save the MLP candidate and its scaler

Saved in Keras's own native format. The scaler is saved separately (joblib) since it is not part of the Keras model itself but is required to preprocess any new input the same way at inference time.


In [7]:
import joblib
model.save("../models/mlp_v2.keras")
joblib.dump(scaler, "../models/mlp_v2_scaler.joblib")
print("saved mlp_v2.keras and mlp_v2_scaler.joblib to ../models/")

saved mlp_v2.keras and mlp_v2_scaler.joblib to ../models/


## Summary

Populated after a local run: validation precision/recall/F1 for the MLP, and how it compares to Random Forest/XGBoost from `04`. Next: `06_evaluation.ipynb` compares all three candidates on the held-out test set and picks the final deployed model.
